In [4]:
from ase.symbols import string2symbols

abinitio_energies = {
        
        'H2_gas': -7.36067,
        'H2O_gas': -14.928449, 
        'C3H6O_gas': -56.022651,
        'C3H6_gas': -48.416361, 
        'H_111':-749.601706,
        'O_111':-751.725634,
        'H2_111':-751.0482531,
        'OH_111':-757.438428,
        'H2O_111': -759.454129,
        'C3H6O_111':-800.435436,
        'C3H5O_111':-801.436942,
        'C3H6_111':-799.775974,
        'O-H2_111':-757.2552457,
        'OH-H_111':-757.776697,
        'C3H5O-H_111':-800.273765,
        'CH3CHOCH-H_111':-799.747906,
        'slab_111': -744.825066,
        }

ref_dict = {}
ref_dict['H'] = abinitio_energies['H2_gas'] *0.5
ref_dict['O'] = abinitio_energies['H2O_gas'] - 2*ref_dict['H']
ref_dict['C'] = (abinitio_energies['C3H6O_gas'] - 6*ref_dict['H']-ref_dict['O'])*1/3
ref_dict['111'] = abinitio_energies['slab_111']

def get_formation_energies(energy_dict,ref_dict):
    formation_energies = {}
    for key in energy_dict.keys(): #iterate through keys
        E0 = energy_dict[key] #raw energy
        name,site = key.split('_') #split key into name/site
        if 'slab' not in name: #do not include empty site energy (0)
            if site == '111':
                E0 -= ref_dict[site] #subtract slab energy if adsorbed
            #remove - from transition-states
            formula = name.replace('-','')
            #get the composition as a list of atomic species
            composition = string2symbols(formula)
            #for each atomic species, subtract off the reference energy
            for atom in composition:
                E0 -= ref_dict[atom]
            #round to 3 decimals since this is the accuracy of DFT
            E0 = round(E0,3)
            formation_energies[key] = E0
    return formation_energies

formation_energies = get_formation_energies(abinitio_energies,ref_dict)
for key in formation_energies:
    print(str(key) + ' ' + str(formation_energies[key]))

H2_gas 0.0
H2O_gas -0.0
C3H6O_gas -0.0
C3H6_gas 0.039
H_111 -1.096
O_111 0.667
H2_111 1.137
OH_111 -1.365
H2O_111 0.299
C3H6O_111 0.412
C3H5O_111 -4.27
C3H6_111 -6.496
O-H2_111 2.498
OH-H_111 1.977
C3H5O-H_111 0.574
CH3CHOCH-H_111 1.1


In [5]:
frequency_dict = {
                
                'H2_gas': [],
                'H2O_gas': [],
                'C3H6O_gas': [],
                'C3H6_gas': [],
                'H_111':[],
                'O_111':[],
                'H2_111':[],
                'OH_111':[],
                'H2O_111': [],
                'C3H6O_111':[],
                'C3H5O_111':[],
                'C3H6_111':[],
                'O-H2_111':[],
                'OH-H_111':[],
                'C3H5O-H_111':[],
                'CH3CHOCH-H_111':[],
                'slab_111': [],
                }
def make_input_file(file_name,energy_dict,frequency_dict):

    #create a header
    header = '\t'.join(['surface_name','site_name',
                        'species_name','formation_energy',
                        'frequencies','reference'])

    lines = [] #list of lines in the output
    for key in energy_dict.keys(): #iterate through keys
        E = energy_dict[key] #raw energy
        name,site = key.split('_') #split key into name/site
        if 'slab' not in name: #do not include empty site energy (0)
            frequency = frequency_dict[key]
            if site == 'gas':
                surface = None
            else:
                surface = 'CeO2-111'
            outline = [surface,site,name,E,frequency,'Input File Tutorial.']
            line = '\t'.join([str(w) for w in outline])
            lines.append(line)

    lines.sort() #The file is easier to read if sorted (optional)
    lines = [header] + lines #add header to top
    input_file = '\n'.join(lines) #Join the lines with a line break

    input = open(file_name,'w') #open the file name in write mode
    input.write(input_file) #write the text
    input.close() #close the file

    print('Successfully created input file')

file_name = 'energies.txt'
make_input_file(file_name,formation_energies,frequency_dict)

Successfully created input file


In [6]:
#Test that input is parsed correctly
from catmap.model import ReactionModel
from catmap.parsers import TableParser
rxm = ReactionModel()
#The following lines are normally assigned by the setup_file
#and are thus not usually necessary.
rxm.surface_names = ['CeO2-111']
rxm.adsorbate_names = ('H','O','H2','OH','H2O','C3H6O','C3H5O','C3H6') 
rxm.transition_state_names = ('O-H2','OH-H','C3H5O-H','CH3CHOCH-H')
rxm.gas_names = ('H2_g','H2O_g','C3H6O_g','C3H6_g')
rxm.site_names = ('s',)
rxm.species_definitions = {'s':{'site_names':['111']}}
#Now we initialize a parser instance (also normally done by setup_file)
parser = TableParser(rxm)
parser.input_file = file_name
parser.parse()
#All structured data is stored in species_definitions; thus we can
#check that the parsing was successful by ensuring that all the
#data in the input file was collected in this dictionary.
for key in rxm.species_definitions:
    print(str(key) + ' ' + str(rxm.species_definitions[key]))

s {'name': 's', 'site': 's', 'type': 'site', 'formation_energy': 0, 'n_sites': 1, 'composition': {}, 'site_names': ['111'], 'frequencies': []}
H2_g {'name': 'H2', 'site': 'g', 'type': 'gas', 'n_sites': 0, 'composition': {'H': 2}, 'formation_energy': 0.0, 'formation_energy_source': 'Input File Tutorial.', 'frequencies': []}
H2O_g {'name': 'H2O', 'site': 'g', 'type': 'gas', 'n_sites': 0, 'composition': {'H': 2, 'O': 1}, 'formation_energy': -0.0, 'formation_energy_source': 'Input File Tutorial.', 'frequencies': []}
C3H6O_g {'name': 'C3H6O', 'site': 'g', 'type': 'gas', 'n_sites': 0, 'composition': {'H': 6, 'C': 3, 'O': 1}, 'formation_energy': -0.0, 'formation_energy_source': 'Input File Tutorial.', 'frequencies': []}
C3H6_g {'name': 'C3H6', 'site': 'g', 'type': 'gas', 'n_sites': 0, 'composition': {'H': 6, 'C': 3}, 'formation_energy': 0.039, 'formation_energy_source': 'Input File Tutorial.', 'frequencies': []}
H {'name': 'H', 'site': 's', 'type': 'adsorbate', 'n_sites': 1, 'composition': {'